In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

meta = pd.read_csv(
    "/content/drive/MyDrive/comm188c/df_900K_consumption_production_final_meta.csv"
)

meta.head()

/tmp/ipykernel_31511/3033994464.py:3: DtypeWarning: Columns (1,2,3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(


,vid,channelId,title,channelTitle,categoryId,tags
0,fd5KQEZVIfg,UCIul30v9pU1JxKx6jlI7wBA,Details on dropped charges against man in vira...,ABC 33/40,25.0,NaN
1,a4xJ-M4eVQc,UCBlXSyoT2PCHp-oMvPIEN6w,There's Something Strange About Ayahuasca,End Times Productions,24.0,"['Tim alberino', 'tim alberino birthrightimoth..."
2,Fg4bcgLsBhg,UC08wrceRCWhajGG4udhETww,Violent Kidnapping Attempt Stopped by Good Sam...,Investigation Discovery,24.0,"['crime', 'crime news', 'real crime', 'crimina..."
3,ovL-CvCfcxg,UCNIFV-Bgh2YtrWQxmrgFS4w,17 Things Frugal People REFUSE to Buy,George Kamel,22.0,"['money', 'budget', 'how to save money', 'pers..."
4,UsUSat0i0as,UCIzKvGZN703WXCbtOANr63A,TIMOTHÉE CHALAMET SURPRISES ZENDAYA WITH HIS A...,dojadog,24.0,"['zendaya', 'zendaya song', 'labrinth zendaya'..."


In [ ]:
channel_counts = (
    meta['channelTitle']
    .value_counts()
    .reset_index()
)

channel_counts.columns = ['channelTitle', 'count']

channel_counts.to_csv(
    '/content/drive/MyDrive/comm188c/channel_counts.csv',
    index=False
)

In [ ]:
selected_channels = [
    'CNN',
    'Fox News',
    'MSNBC',
    'ABC News',
    'NBC News',
    'CBS News',
    'NewsNation',
    'Sky News Australia'
]

news_meta_8 = meta[
    meta['channelTitle'].isin(selected_channels)
]

print(news_meta_8.shape)

news_meta_8['channelTitle'].value_counts()

(23333, 6)


,count
channelTitle,
MSNBC,5259
Fox News,3850
NBC News,2891
ABC News,2605
CBS News,2346
NewsNation,2216
CNN,2106
Sky News Australia,2060


In [ ]:
weather_keywords = (
    "hurricane|typhoon|cyclone|tropical storm|storm surge|"
    "flood|flooding|flash flood|flood warning|"
    "tornado|tornado warning|"
    "wildfire|forest fire|bushfire|"
    "heatwave|heat wave|extreme heat|heat dome|heat index|"
    "record heat|record temperature|"
    "drought|"
    "blizzard|ice storm|freezing rain|snowfall|winter storm|"
    "cold wave|cold snap|wind chill|"
    "atmospheric river|el nino|la nina|"
    "severe weather|extreme weather|weather warning|weather alert|"
    "weather advisory|weather watch|"
    "heavy rain|torrential rain|hail|lightning|thunderstorm|"
    "national weather service|meteorologist|noaa"
)

weather_news_8 = news_meta_8[
    news_meta_8["title"].str.contains(
        weather_keywords,
        case=False,
        na=False,
        regex=True
    )
].copy()

print(weather_news_8.shape)

weather_news_8["channelTitle"].value_counts()

(449, 6)


,count
channelTitle,
CBS News,111
NBC News,105
ABC News,97
MSNBC,35
NewsNation,32
CNN,26
Fox News,24
Sky News Australia,19


In [ ]:
weather_news_8[["channelTitle", "title"]].sample(
    30,
    random_state=123
)

,channelTitle,title
94803,NBC News,Massive tornado outbreak reduced areas to rubb...
152297,ABC News,Wildfires rage across the West as California f...
58315,CBS News,"Heat dome expands to Midwest, Northeast U.S."
926792,CBS News,Hurricane Helene expected to be one of the cos...
939106,CBS News,Birds caught in eye of Hurricane Helene #shorts
833118,CBS News,Helene becomes hurricane before Florida landfall
214168,Fox News,Florida sheriff: 'We've never seen this many t...
163040,ABC News,Record-breaking floods in Minnesota cause brok...
1117936,NBC News,Florida residents face massive cleanup after H...
28074,NBC News,4-month-old baby found safe in tree after Tenn...


In [ ]:
# Step 1: get video IDs from weather news
relevant_vids = set(weather_news_8["vid"])

print("Weather news videos:", len(relevant_vids))


# Step 2: match transcripts from vid_yt.csv
chunks = []

for chunk in pd.read_csv(
    "/content/drive/MyDrive/comm188c/vid_yt.csv",
    chunksize=100000
):
    filtered = chunk[
        chunk["vid"].isin(relevant_vids)
    ]
    chunks.append(filtered)


# Step 3: combine matched transcript chunks
vid_filtered_8 = pd.concat(chunks)

print("Matched transcript rows:", vid_filtered_8.shape)
print("Unique matched vids:", vid_filtered_8["vid"].nunique())

vid_filtered_8.head()

Weather news videos: 449
Matched transcript rows: (222, 2)
Unique matched vids: 222


,vid,cc_segment
14,DkbNsSuA3kk,italian chief actually just got out out of his...
513,5XSQykEV_mY,overall . there 's still million gallons of di...
2102,SsqRj_buV5c,i will have more on the coming up in a bit but...
2255,a01puB8A0p4,morphine . hundreds of times stronger than her...
3071,0h3fF2rsZGo,that money go ? 've also asked the white house...


In [ ]:
final_weather_8 = pd.merge(
    weather_news_8,
    vid_filtered_8,
    on="vid",
    how="inner"
)

print(final_weather_8.shape)

final_weather_8.head()

(222, 7)


,vid,channelId,title,channelTitle,categoryId,tags,cc_segment
0,r6RuUbI618M,UCCjG8NtOig0USdrT5D1FpxQ,"'As far as intensity, it’s up to Mother Nature...",NewsNation,25.0,"['WatchTheHill', 'TheHill', 'Hurricane Helene'...","53rd whether a scroll to the candidates , the ..."
1,tyjqNHIrWWk,UCO0akufu9MOzyz3nvGIXAAw,At least 19 dead amid deadly flooding in Europe,Sky News Australia,25.0,"['6362086577112', 'fb', 'global', 'msn', 'worl...",mass evacuation orders have been issued due to...
2,TuifNWqITxk,UCCjG8NtOig0USdrT5D1FpxQ,Fort Myers resident hopeful in wake of Hurrica...,NewsNation,25.0,"['Hurricane Milton', 'Fort Myers', 'Milton', '...",enough to where they can go out and they could...
3,Ky8kCs0g4Ok,UCeY0bbntWzzVIaj2z3QigXg,Bermuda braces for Hurricane Ernesto,NBC News,25.0,"['Health', 'International News', 'Lester Holt'...",starting monday for convention coverage live f...
4,PB38k1e6624,UCXIJgqnII2ZOINSWNOGFThA,Florida sheriff: 'We've never seen this many t...,Fox News,25.0,"['breaking news', 'florida hurricane live', 'f...",remarkable how many tornado warnings we had ov...


In [ ]:
final_weather_8.to_csv(
    "/content/drive/MyDrive/comm188c/final_weather_8media.csv",
    index=False
)

In [ ]:
final_weather_8[
    ["channelTitle","title","cc_segment"]
].sample(20, random_state=123)

,channelTitle,title,cc_segment
188,ABC News,ABC News Prime: Deadly tornadoes hit 4 states;...,we 'll take you there . > > you 're streaming ...
213,CBS News,Largest wildfire in Texas history burns 1.1 mi...,between israeli troops and palestinians waitin...
71,CBS News,Tennessee family trapped in home for hours as ...,oh i hope not oh we 're going to go up in the ...
20,MSNBC,FEMA preparing for 'multi-state event' as Hurr...,heee than any storm previous on record . > > b...
79,CBS News,How Hurricane Helene is affecting the vote in ...,let 's talk about hurricane recovery efforts h...
179,ABC News,Houston mayor gives update on Hurricane Beryl ...,we have breaking news right now the houston of...
202,ABC News,Tracking extreme weather across the nation,that one store michael is a powerhouse waking ...
165,NBC News,Climate leaders discuss extreme heat crisis an...,so much for being here . hope you had a nice l...
211,CBS News,Thousands ordered to evacuate as California wi...,> > > it 's 10:00 am on the east coast . 7 am ...
90,ABC News,92 unaccounted for in North Carolina after Hur...,we turn next tonight to two new tropical threa...
